### 1. Refactored Password Manager Backend

I've refactored your existing password manager code into a `PasswordManager` class. This class encapsulates all the logic, making it easier to manage and integrate with a user interface. Key changes include:

*   Methods like `set_master_password`, `verify_master_password`, `view_passwords`, and `add_password` no longer use `input()` or `print()`. Instead, they take arguments directly and return results (e.g., `(success, message)` or `(data, message)`), which is ideal for UI interaction.
*   The `Fernet` cipher is now initialized as an instance variable (`self.fernet_cipher`) only after the master password has been successfully set or verified, ensuring that cryptographic operations are only performed when the user is authenticated.

In [ ]:
import cryptography
from cryptography.fernet import Fernet
import os
import hashlib

class PasswordManager:
    MASTER_PWD_FILE = "master_pwd.key"
    KEY_FILE = "key.key"

    def __init__(self):
        self.fernet_cipher = None  # Will be initialized after master password verification
        self.fernet_key = None
        self._load_or_create_key()

    def _write_key(self):
        key = Fernet.generate_key()
        with open(self.KEY_FILE, "wb") as key_file:
            key_file.write(key)

    def _load_key(self):
        if not os.path.exists(self.KEY_FILE):
            self._write_key()
        with open(self.KEY_FILE, "rb") as file:
            key = file.read()
        return key

    def _load_or_create_key(self):
        """Ensures the Fernet key exists and loads it."""
        try:
            self.fernet_key = self._load_key()
        except FileNotFoundError:
            # This case should ideally be handled by _load_key creating the file
            self._write_key()
            self.fernet_key = self._load_key()

    @staticmethod
    def hash_password(pwd):
        return hashlib.sha256(pwd.encode()).hexdigest()

    def set_master_password(self, new_master_pwd, confirm_master_pwd):
        if not new_master_pwd or not confirm_master_pwd:
            return False, "Master password fields cannot be empty."
        if new_master_pwd == confirm_master_pwd:
            with open(self.MASTER_PWD_FILE, "w") as f:
                f.write(self.hash_password(new_master_pwd))
            self.fernet_cipher = Fernet(self.fernet_key)  # Initialize cipher on success
            return True, "Master password set successfully!"
        else:
            return False, "Passwords do not match. Please try again."

    def verify_master_password(self, entered_master_pwd):
        if not entered_master_pwd:
            return False, "Please enter a master password."

        if not os.path.exists(self.MASTER_PWD_FILE):
            return False, "No master password found. Please set one first."

        with open(self.MASTER_PWD_FILE, "r") as f:
            stored_hash = f.read().strip()

        if self.hash_password(entered_master_pwd) == stored_hash:
            self.fernet_cipher = Fernet(self.fernet_key)  # Initialize cipher on success
            return True, "Master password verified."
        else:
            return False, "Incorrect master password."

    def view_passwords(self):
        if not self.fernet_cipher:
            return None, "Master password not verified. Please verify to view passwords."
        if not os.path.exists('passwords.txt') or os.stat('passwords.txt').st_size == 0:
            return [], "No passwords saved yet."

        passwords_list = []
        with open('passwords.txt', 'r') as f:
            for line in f.readlines():
                if not line.strip():
                    continue
                try:
                    data = line.rstrip()
                    user, enc_passw = data.split("|")
                    decrypted_passw = self.fernet_cipher.decrypt(enc_passw.encode()).decode()
                    passwords_list.append({"user": user, "password": decrypted_passw})
                except Exception as e:
                    # Log error but don't stop processing other entries
                    print(f"Error decrypting line: {line.rstrip()} - {e}")
        return passwords_list, ""  # Return list and empty message on success

    def add_password(self, name, pwd):
        if not self.fernet_cipher:
            return False, "Master password not verified. Please verify to add passwords."
        if not name or not pwd:
            return False, "Account name and password cannot be empty."

        try:
            encrypted_pwd = self.fernet_cipher.encrypt(pwd.encode()).decode()
            with open('passwords.txt', 'a') as f:
                f.write(f"{name}|{encrypted_pwd}\n")
            return True, "Password added successfully!"
        except Exception as e:
            return False, f"Error adding password: {e}"


### 2. Interactive UI with `ipywidgets`

This section creates an interactive user interface using `ipywidgets`. The UI is divided into two main parts:

*   **Authentication Area**: Where you'll enter or set your master password.
*   **Main Application Area**: This area will become visible only after successful master password verification, allowing you to add or view your stored passwords.

Instructions:
1.  If you're running this for the first time or if `master_pwd.key` does not exist, use the "Set New Master Password" button to create one.
2.  Otherwise, enter your master password and click "Verify Master Password" to unlock the application.
3.  Once unlocked, you can use the "Add Password" and "View Passwords" buttons.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Instantiate the PasswordManager
pm = PasswordManager()

# --- UI Elements ---
master_pwd_input = widgets.Password(
    description='Master PWD:',
    placeholder='Enter master password',
    layout=widgets.Layout(width='auto')
)
confirm_master_pwd_input = widgets.Password(
    description='Confirm PWD:',
    placeholder='Confirm master password',
    layout=widgets.Layout(width='auto')
)

verify_button = widgets.Button(description='Verify Master Password', button_style='info')
set_button = widgets.Button(description='Set New Master Password', button_style='warning')

auth_output = widgets.Output()  # For displaying authentication messages

# Main password management UI elements (initially hidden)
account_name_input = widgets.Text(description='Account:', placeholder='e.g., Google', disabled=True)
account_pwd_input = widgets.Password(description='Password:', placeholder='Enter password', disabled=True)
add_pwd_button = widgets.Button(description='Add Password', button_style='success', disabled=True)
view_pwd_button = widgets.Button(description='View Passwords', button_style='primary', disabled=True)

main_app_output = widgets.Output()  # For displaying password manager results

# --- Layouts ---
master_pwd_box = widgets.VBox([
    master_pwd_input,
    confirm_master_pwd_input,  # This input is only relevant for setting
    widgets.HBox([verify_button, set_button]),
    auth_output
])

main_app_box = widgets.VBox([
    account_name_input,
    account_pwd_input,
    widgets.HBox([add_pwd_button, view_pwd_button]),
    main_app_output
])

# Initially hide the main app until master password is verified
main_app_box.layout.visibility = 'hidden'

# --- Event Handlers ---
def update_main_app_visibility(authenticated):
    account_name_input.disabled = not authenticated
    account_pwd_input.disabled = not authenticated
    add_pwd_button.disabled = not authenticated
    view_pwd_button.disabled = not authenticated
    main_app_box.layout.visibility = 'visible' if authenticated else 'hidden'

def on_verify_button_clicked(b):
    with auth_output:
        clear_output()
        master_pwd = master_pwd_input.value

        success, message = pm.verify_master_password(master_pwd)
        if success:
            print(HTML(f'<span style="color:green;">{message}</span>'))
        else:
            print(HTML(f'<span style="color:red;">{message}</span>'))
        update_main_app_visibility(success)
        # Clear sensitive input after attempt
        master_pwd_input.value = ''
        confirm_master_pwd_input.value = ''

def on_set_button_clicked(b):
    with auth_output:
        clear_output()
        new_pwd = master_pwd_input.value
        confirm_pwd = confirm_master_pwd_input.value

        success, message = pm.set_master_password(new_pwd, confirm_pwd)
        if success:
            print(HTML(f'<span style="color:green;">{message}</span>'))
        else:
            print(HTML(f'<span style="color:red;">{message}</span>'))
        update_main_app_visibility(success)
        # Clear sensitive input after attempt
        master_pwd_input.value = ''
        confirm_master_pwd_input.value = ''

def on_add_pwd_button_clicked(b):
    with main_app_output:
        clear_output()
        name = account_name_input.value
        pwd = account_pwd_input.value

        success, message = pm.add_password(name, pwd)
        if success:
            print(HTML(f'<span style="color:green;">{message}</span>'))
            # Clear inputs after adding
            account_name_input.value = ''
            account_pwd_input.value = ''
        else:
            print(HTML(f'<span style="color:red;">{message}</span>'))

def on_view_pwd_button_clicked(b):
    with main_app_output:
        clear_output()
        passwords, msg = pm.view_passwords()
        if passwords is None:  # Master password not verified error
            print(HTML(f'<span style="color:red;">{msg}</span>'))
            update_main_app_visibility(False)  # Re-hide if somehow unverified
            return
        elif not passwords:  # No passwords saved yet
            print(HTML(f'<span style="color:orange;">{msg}</span>'))
            return

        # Display passwords in a more structured way
        display(HTML("<h3>Your Stored Passwords:</h3>"))
        for item in passwords:
            display(HTML(f"<b>Account:</b> {item['user']} | <b>Password:</b> {item['password']}"))


# --- Attach Handlers ---
verify_button.on_click(on_verify_button_clicked)
set_button.on_click(on_set_button_clicked)
add_pwd_button.on_click(on_add_pwd_button_clicked)
view_pwd_button.on_click(on_view_pwd_button_clicked)

# --- Display UI ---
display(master_pwd_box, main_app_box)
